# DeBCR API tutorial
## Train DeBCR model on pre-processed data

This notebook shows how to train DeBCR model to restore low-quality microscopy data.

To achieve that you need **pre-processed training/validation data**, consisting of low-quality input and high-quality ground truth, both normalized and patched.

Please find on the DeBCR GitHub page links to:
- notebook tutorial on raw data pre-processing protocol; 
- samples, i.e. examples of pre-processed training/validation data.

In [ ]:
import debcr

### Load training/validation data

Set file path to your actual pre-processed input data (in NPZ or NPY format).

For sample data:
- training data: ```/path/to/examples/DATASET/data/DATASET_train.npz```
- validation data: ```/path/to/examples/DATASET/data/DATASET_val.npz```

In [ ]:
train_data_filepath = '/path/to/load/data/train.npz'
val_data_filepath = '/path/to/load/data/val.npz'
train_data_filepath, val_data_filepath

Load training/validation data

In [ ]:
data_train = debcr.data.load(train_data_filepath)
data_val = debcr.data.load(val_data_filepath)

The example training/validaton data is provided as multi-array NPZ, which contains two arrays:
- "low" - input data (low-quality data to be improved)
- "gt" - ground-truth data for comparison

You can check the filenames as below:

In [ ]:
data_train.files, data_val.files

### Visualize loaded training/validation data

for sample training data

In [ ]:
debcr.data.show(
    data = [data_train["low"], data_train["gt"]],
    slices = [-1, -1, -1], # -1 is to pick a random slice
    titles = ['train: input', 'train: ground truth'],
    transpose = True
)

for sample validation data

In [ ]:
debcr.data.show(
    data = [data_val["low"], data_val["gt"]],
    slices = [-1, -1, -1], # -1 is to pick a random slice
    titles = ['validation: input', 'validation: ground truth'],
    transpose = True
)

### Setup model to train

You can either start training from scratch or continue training some existing model. 

#### a. Initialize new model

To start training of the DeBCR model from scratch, setup a new model (provide the requested model input size):

In [ ]:
start_model = debcr.model.init(input_size=128)

#### b. Load existing model

To continue training the existing pre-trained DeBCR model, load this model from drive:

In [ ]:
start_model = debcr.model.init(weights_path='/path/to/load/model/weights', input_size=128)

Additionally, you may check the model layers and parameters information

In [ ]:
start_model.summary()

### Setup training configuration

Next, we need to set the configuration parameters for the training.

#### a. Load default training configuration

In [ ]:
config = debcr.config.load()
config

#### b. Load training configuration from YAML file

In [ ]:
config_path = '/path/to/load/config.yaml'
config = debcr.config.load(config_path)
config

Next, provide the path to save the output model you are going to train: 

In [ ]:
config['weights_path'] = '/path/to/save/trained/model/weights'
config

You can adjust other configuration parameters as needed, for example

In [ ]:
config['batch_size'] = 16
config

Additionally, you can save the latest version of this training configuration as YAML file

In [ ]:
config_path = '/path/to/save/config.yaml'
debcr.config.save(config, config_path)

### Train model on training data

To start training the DeBCR model, pass to the dedicated training interface:
- the starting model (new or existing);
- the training/validation data;
- the training configuration.

Intermediate checkpoints will be printed below.

The training will stop automatically upon the model convergence or defined number of the training iterations being achieved.

In [ ]:
debcr_model = debcr.model.train(data_train, data_val, config, start_model)

### Test trained model on validation data

Finally, we can try to run the freshly trained model on the validation data

In [ ]:
data_pred = debcr.model.predict(debcr_model, data_val["low"])
data_pred.shape

In [ ]:
debcr.data.show(
    data = [data_val['low'], data_pred, data_val["gt"]],
    slices = [-1, -1],
    titles = ['validation input', 'validation prediction', 'validation ground truth']
)

Further you can use the trained here model to run predictions (see the tutorial link on the GitHub).